# Notebook 04 — Demo del Agente de Contratos
## Instrucciones para Reproducir (leer antes de ejecutar)

**Paso 1:** Guardar una copia de este notebook en su Drive
> Archivo → Guardar una copia en Drive

**Paso 2:** Agregar la carpeta del adapter a su Drive
> Abrir el link compartido → Click derecho → "Agregar acceso directo a Drive"
> Guardarlo en: Mi unidad/agente-contratos-cto/adapter

**Paso 3:** Agregar el token de Databricks en Colab Secrets
> Panel izquierdo → ícono 🔑 → Nuevo secreto
> Nombre: DATABRICKS_TOKEN | Valor: su token de databricks.com (cuenta gratuita)

**Paso 4:** Seleccionar GPU como entorno de ejecución
> Entorno de ejecución → Cambiar tipo de entorno → GPU T4

**Paso 5:** Ejecutar todo
> Entorno de ejecución → Ejecutar todo (Ctrl+F9)

In [ ]:
# Celda 2 — Instalación de dependencias
!pip install -q torch==2.3.0 transformers==4.44.0 peft==0.12.0 bitsandbytes==0.43.3 langchain==0.2.16 langchain-community==0.2.16 accelerate==0.33.0 sentencepiece==0.2.0

In [ ]:
# Celda 3 — Verificar GPU, montar Drive y verificar adapter
import torch

assert torch.cuda.is_available(), (
    "⚠️ Activar GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU T4"
)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

RUTA_ADAPTER = Path('/content/drive/MyDrive/agente-contratos-cto/adapter')
assert RUTA_ADAPTER.exists(), (
    f"⚠️ Adapter no encontrado en {RUTA_ADAPTER}. Revisar Paso 2 de las instrucciones."
)

print("✅ GPU disponible y adapter encontrado")

## Carga del Modelo Fine-Tuneado

In [ ]:
# Celda 5 — Cargar modelo base con cuantización 4-bit y fusionar adapter LoRA
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)
from peft import PeftModel
from langchain_community.llms import HuggingFacePipeline

# Nombre del modelo base
NOMBRE_MODELO_BASE: str = "Qwen/Qwen2.5-7B-Instruct"

# Configuración de cuantización a 4 bits para reducir consumo de VRAM
configuracion_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Cargando tokenizador...")
tokenizador = AutoTokenizer.from_pretrained(
    NOMBRE_MODELO_BASE,
    trust_remote_code=True,
)
tokenizador.pad_token = tokenizador.eos_token

print("Cargando modelo base con cuantización 4-bit...")
modelo_base = AutoModelForCausalLM.from_pretrained(
    NOMBRE_MODELO_BASE,
    quantization_config=configuracion_bnb,
    device_map="auto",
    trust_remote_code=True,
)

print("Fusionando adapter LoRA...")
modelo_con_adapter = PeftModel.from_pretrained(
    modelo_base,
    str(RUTA_ADAPTER),
)
modelo_fusionado = modelo_con_adapter.merge_and_unload()
print("✅ Modelo fusionado correctamente")

# Crear pipeline de generación de texto
tuberia_texto = pipeline(
    task="text-generation",
    model=modelo_fusionado,
    tokenizer=tokenizador,
    max_new_tokens=1024,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.15,
    do_sample=True,
)

# Envolver en LangChain para uso con agentes
llm = HuggingFacePipeline(pipeline=tuberia_texto)
print("✅ Pipeline de LangChain lista")

## Definición de Herramientas del Agente

In [ ]:
# Celda 7 — Definir las 4 herramientas del agente
from langchain.tools import tool
from typing import List

# Aviso legal que se agrega al final de cada respuesta generada
AVISO_LEGAL: str = (
    "\n\n⚠️ Este análisis es asistido por IA y no constituye "
    "asesoría legal profesional."
)

# Prompt de sistema compartido por todas las herramientas
PROMPT_SISTEMA: str = (
    "Eres un agente experto en análisis de contratos tecnológicos para CTOs."
)


def _invocar_modelo(mensaje_usuario: str) -> str:
    """Formatea la entrada como chat, invoca la pipeline y devuelve la respuesta.

    Args:
        mensaje_usuario: Texto del mensaje del usuario para el modelo.

    Returns:
        Texto generado por el modelo con el aviso legal anexado.
    """
    mensajes: List[dict] = [
        {"role": "system", "content": PROMPT_SISTEMA},
        {"role": "user", "content": mensaje_usuario},
    ]
    texto_formateado: str = tokenizador.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True,
    )
    salida = tuberia_texto(
        texto_formateado,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
    )
    respuesta: str = salida[0]["generated_text"]
    # Extraer solo la parte generada (después del prompt)
    if texto_formateado in respuesta:
        respuesta = respuesta[len(texto_formateado):].strip()
    return respuesta + AVISO_LEGAL


@tool
def analizar_clausula(texto_clausula: str) -> str:
    """Analiza una cláusula contractual e identifica riesgos para la empresa compradora."""
    consulta: str = (
        f"Analiza la siguiente cláusula contractual. Identifica todos los "
        f"riesgos para la empresa compradora, clasifícalos por severidad "
        f"(alta, media, baja) y proporciona una recomendación ejecutiva.\n\n"
        f"Cláusula:\n{texto_clausula}"
    )
    return _invocar_modelo(consulta)


@tool
def comparar_propuestas(propuesta_a: str, propuesta_b: str) -> str:
    """Compara dos propuestas de vendors y recomienda la mejor opción."""
    consulta: str = (
        f"Compara las siguientes dos propuestas de proveedores tecnológicos. "
        f"Evalúa cada una con un puntaje de 1 a 10 en las dimensiones de: "
        f"costo, escalabilidad, soporte, seguridad y cumplimiento normativo. "
        f"Proporciona una decisión final justificada.\n\n"
        f"--- Propuesta A ---\n{propuesta_a}\n\n"
        f"--- Propuesta B ---\n{propuesta_b}"
    )
    return _invocar_modelo(consulta)


@tool
def alerta_renovacion(metadata_contrato: str) -> str:
    """Evalúa un contrato próximo a vencer y genera un plan de acción."""
    consulta: str = (
        f"Evalúa el siguiente contrato próximo a vencer. Determina el nivel "
        f"de urgencia, calcula la fecha límite de decisión e incluye un "
        f"checklist de acciones previas a la renovación.\n\n"
        f"Metadata del contrato:\n{metadata_contrato}"
    )
    return _invocar_modelo(consulta)


@tool
def negociar(contexto_contrato: str) -> str:
    """Genera una estrategia de negociación basada en el contexto del contrato."""
    consulta: str = (
        f"Genera una estrategia de negociación completa para el siguiente "
        f"contexto contractual. Incluye: BATNA (mejor alternativa al acuerdo "
        f"negociado), 3 tácticas concretas de negociación y las líneas rojas "
        f"que no deben cruzarse.\n\n"
        f"Contexto del contrato:\n{contexto_contrato}"
    )
    return _invocar_modelo(consulta)


# Lista de herramientas disponibles para el agente
lista_herramientas = [analizar_clausula, comparar_propuestas, alerta_renovacion, negociar]
print(f"✅ {len(lista_herramientas)} herramientas definidas:")
for herramienta in lista_herramientas:
    print(f"   - {herramienta.name}: {herramienta.description}")

## Configuración del Agente ReAct

In [ ]:
# Celda 9 — Crear agente ReAct con LangChain
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate

# Plantilla del prompt ReAct en español
PLANTILLA_REACT: str = """Eres un agente experto en análisis de contratos tecnológicos para CTOs.
Responde en español de forma clara, estructurada y profesional.

Tienes acceso a las siguientes herramientas:

{tools}

Usa el siguiente formato:

Pregunta: la pregunta de entrada que debes responder
Pensamiento: siempre debes pensar qué hacer a continuación
Acción: la acción a tomar, debe ser una de [{tool_names}]
Entrada de Acción: la entrada para la acción
Observación: el resultado de la acción
... (este ciclo Pensamiento/Acción/Entrada de Acción/Observación puede repetirse N veces)
Pensamiento: Ya tengo la respuesta final
Respuesta Final: la respuesta final a la pregunta original

¡Comienza!

Pregunta: {input}
Pensamiento: {agent_scratchpad}"""

plantilla_prompt = PromptTemplate(
    template=PLANTILLA_REACT,
    input_variables=["input", "agent_scratchpad"],
    partial_variables={
        "tools": "\n".join(
            [f"- {h.name}: {h.description}" for h in lista_herramientas]
        ),
        "tool_names": ", ".join([h.name for h in lista_herramientas]),
    },
)

# Crear el agente ReAct
agente_react = create_react_agent(
    llm=llm,
    tools=lista_herramientas,
    prompt=plantilla_prompt,
)

# Crear el ejecutor del agente con manejo de errores
ejecutor_agente = AgentExecutor(
    agent=agente_react,
    tools=lista_herramientas,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=6,
    return_intermediate_steps=True,
)

print("✅ Agente ReAct configurado y listo para ejecutar")

## Caso 1: Análisis de Cláusula de Renovación Automática

In [ ]:
# Celda 11 — Caso de uso 1: Analizar cláusula de renovación automática

# Cláusula sintética de ejemplo
clausula_renovacion: str = (
    "Cláusula 8.1 - Renovación Automática: El presente contrato se renovará "
    "automáticamente por períodos sucesivos de 12 meses, a menos que "
    "cualquiera de las partes notifique por escrito su intención de no "
    "renovar con un mínimo de 90 días de anticipación a la fecha de "
    "vencimiento. En caso de renovación automática, el proveedor se reserva "
    "el derecho de ajustar las tarifas hasta en un 15% sin necesidad de "
    "consentimiento previo del cliente. La penalización por cancelación "
    "anticipada posterior a la renovación será equivalente al 40% del valor "
    "anual restante del contrato."
)

print("=" * 70)
print("CASO 1: ANÁLISIS DE CLÁUSULA DE RENOVACIÓN AUTOMÁTICA")
print("=" * 70)
print(f"\nCláusula a analizar:\n{clausula_renovacion}\n")
print("-" * 70)

# Invocación directa de la herramienta para mayor confiabilidad
resultado_caso_1: str = analizar_clausula.invoke({"texto_clausula": clausula_renovacion})

print("\nRESULTADO DEL ANÁLISIS:")
print("-" * 70)
print(resultado_caso_1)

## Caso 2: Comparación de Propuestas AWS vs Azure

In [ ]:
# Celda 13 — Caso de uso 2: Comparar propuestas AWS vs Azure

# Propuestas sintéticas de ejemplo
propuesta_aws: str = (
    "Propuesta AWS — Migración de Infraestructura Bancaria\n"
    "Proveedor: Amazon Web Services\n"
    "Costo anual estimado: USD 480,000\n"
    "Duración del contrato: 3 años con reserva de instancias\n"
    "SLA de disponibilidad: 99.99%\n"
    "Soporte: Enterprise Support 24/7 con TAM dedicado\n"
    "Seguridad: Cumplimiento PCI-DSS, SOC 2 Type II, ISO 27001\n"
    "Migración: Incluye equipo de Professional Services (90 días)\n"
    "Escalabilidad: Auto Scaling ilimitado con instancias reservadas\n"
    "Penalización por salida anticipada: 25% del valor restante"
)

propuesta_azure: str = (
    "Propuesta Azure — Migración de Infraestructura Bancaria\n"
    "Proveedor: Microsoft Azure\n"
    "Costo anual estimado: USD 520,000\n"
    "Duración del contrato: 3 años con Azure Reserved Instances\n"
    "SLA de disponibilidad: 99.95%\n"
    "Soporte: Unified Support Premier con CSAM dedicado\n"
    "Seguridad: Cumplimiento PCI-DSS, SOC 2 Type II, ISO 27001, "
    "integración nativa con Active Directory\n"
    "Migración: Azure Migrate + FastTrack (120 días)\n"
    "Escalabilidad: Virtual Machine Scale Sets con reservas\n"
    "Penalización por salida anticipada: 20% del valor restante\n"
    "Beneficio adicional: Licencias Microsoft 365 E3 incluidas "
    "para 200 usuarios"
)

print("=" * 70)
print("CASO 2: COMPARACIÓN DE PROPUESTAS AWS vs AZURE")
print("=" * 70)
print(f"\n{propuesta_aws}\n")
print(f"{propuesta_azure}\n")
print("-" * 70)

# Invocación directa de la herramienta
resultado_caso_2: str = comparar_propuestas.invoke({
    "propuesta_a": propuesta_aws,
    "propuesta_b": propuesta_azure,
})

print("\nRESULTADO DE LA COMPARACIÓN:")
print("-" * 70)
print(resultado_caso_2)

## Caso 3: Alerta de Renovación de Contrato CRM

In [ ]:
# Celda 15 — Caso de uso 3: Alerta de renovación de contrato CRM

# Metadata sintética de contrato SaaS CRM
metadata_crm: str = (
    "Contrato: Licencia SaaS CRM Enterprise\n"
    "Proveedor: CRM Solutions Inc.\n"
    "Fecha de inicio: 2023-06-15\n"
    "Fecha de vencimiento: 2026-04-05\n"
    "Días hasta vencimiento: 45 días\n"
    "Gasto anual: USD 120,000\n"
    "Usuarios activos: 350\n"
    "Cláusula de renovación: Automática por 12 meses si no se "
    "notifica 60 días antes\n"
    "Incremento previsto: 12% sobre tarifa actual\n"
    "Alternativas evaluadas: Salesforce, HubSpot Enterprise\n"
    "Nivel de satisfacción interna: 6.5/10\n"
    "Incidentes reportados último año: 8 (3 críticos)"
)

print("=" * 70)
print("CASO 3: ALERTA DE RENOVACIÓN DE CONTRATO CRM")
print("=" * 70)
print(f"\n{metadata_crm}\n")
print("-" * 70)

# Invocación directa de la herramienta
resultado_caso_3: str = alerta_renovacion.invoke({"metadata_contrato": metadata_crm})

print("\nRESULTADO DE LA ALERTA:")
print("-" * 70)
print(resultado_caso_3)

## Ejecución del Agente Completo (ReAct)

In [ ]:
# Celda 17 — Ejecución completa del agente ReAct con consulta multiherramienta

# Consulta compleja que requiere razonamiento y uso de múltiples herramientas
consulta_compleja: str = (
    "Tenemos un contrato de infraestructura cloud con AWS que vence en 45 días "
    "por un valor de USD 500,000 anuales. El contrato tiene una cláusula de "
    "renovación automática de 12 meses con aviso de 90 días y penalización "
    "del 30% por cancelación anticipada. Recibimos una contrapropuesta de "
    "Azure por USD 520,000 anuales pero con licencias M365 incluidas y "
    "mejor integración con nuestro Active Directory. Necesito que: "
    "1) Analices los riesgos de la cláusula de renovación automática, "
    "2) Compares ambas propuestas, y "
    "3) Generes una estrategia de negociación para obtener mejores condiciones "
    "con el proveedor actual antes de decidir migrar."
)

print("=" * 70)
print("EJECUCIÓN DEL AGENTE REACT — CONSULTA MULTIHERRAMIENTA")
print("=" * 70)
print(f"\nConsulta:\n{consulta_compleja}\n")
print("=" * 70)
print("TRAZA DE RAZONAMIENTO DEL AGENTE:")
print("=" * 70)

# Ejecutar el agente (verbose=True muestra la traza ReAct completa)
resultado_agente = ejecutor_agente.invoke({"input": consulta_compleja})

print("\n" + "=" * 70)
print("RESPUESTA FINAL DEL AGENTE:")
print("=" * 70)
print(resultado_agente["output"])

# Mostrar los pasos intermedios del agente
if resultado_agente.get("intermediate_steps"):
    print("\n" + "=" * 70)
    print("PASOS INTERMEDIOS EJECUTADOS:")
    print("=" * 70)
    for indice, (accion, observacion) in enumerate(
        resultado_agente["intermediate_steps"], start=1
    ):
        print(f"\nPaso {indice}:")
        print(f"  Herramienta: {accion.tool}")
        print(f"  Entrada: {accion.tool_input}")
        print(f"  Observación: {observacion[:200]}...")

## Resumen de Capacidades del Agente

| Herramienta | Entrada | Salida esperada |
|-------------|---------|------------------|
| `analizar_clausula` | Texto de cláusula contractual | Riesgos con severidad (alta/media/baja) + recomendación ejecutiva |
| `comparar_propuestas` | Propuesta A y B en texto libre | Puntaje 1-10 por dimensión + decisión final justificada |
| `alerta_renovacion` | Metadata del contrato (fechas, valor, proveedor) | Nivel de urgencia + fecha límite de decisión + checklist |
| `negociar` | Contexto del contrato + objetivos del CTO | BATNA + 3 tácticas concretas + líneas rojas |

⚠️ **Aviso Legal:** Todos los análisis son asistidos por IA y no constituyen asesoría legal profesional.

## Interfaz Interactiva con Gradio

Interfaz web para probar el agente en vivo. Permite:
- **Subir un contrato** en formato `.txt` o `.pdf` y analizarlo con cualquiera de las herramientas
- **Comparar dos propuestas** pegando el texto de cada una
- **Hacer consultas libres** al agente ReAct, que encadena herramientas automáticamente

> Al ejecutar la celda se genera un link público temporal (válido por 72 horas) para compartir la demo.

In [ ]:
!pip install -q gradio==4.44.0 PyPDF2==3.0.1

In [ ]:
import gradio as gr
from PyPDF2 import PdfReader


def extraer_texto_archivo(archivo) -> str:
    """Extrae texto plano de un archivo .txt o .pdf subido por el usuario.

    Args:
        archivo: Ruta al archivo temporal subido por Gradio.

    Returns:
        Texto extraído del archivo, o cadena vacía si no se proporcionó archivo.
    """
    if archivo is None:
        return ""

    ruta: str = archivo if isinstance(archivo, str) else archivo.name

    if ruta.lower().endswith(".pdf"):
        lector = PdfReader(ruta)
        paginas: list[str] = [
            pagina.extract_text() or "" for pagina in lector.pages
        ]
        return "\n".join(paginas).strip()

    # Asumir texto plano para cualquier otro formato
    with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()


# ---------------------------------------------------------------------------
# Funciones de callback para cada pestaña de Gradio
# ---------------------------------------------------------------------------

def cb_analizar(archivo, texto_manual: str, herramienta: str) -> str:
    """Callback para la pestaña 'Analizar Contrato'.

    Extrae texto del archivo subido o usa el texto manual, y lo procesa
    con la herramienta seleccionada.

    Args:
        archivo: Archivo subido (.txt o .pdf), puede ser None.
        texto_manual: Texto ingresado manualmente por el usuario.
        herramienta: Nombre de la herramienta a utilizar.

    Returns:
        Resultado del análisis generado por el modelo.
    """
    texto_archivo: str = extraer_texto_archivo(archivo)
    texto: str = texto_archivo if texto_archivo else texto_manual.strip()

    if not texto:
        return "Ingrese texto o suba un archivo para analizar."

    mapa_herramientas: dict = {
        "Analizar Clausula": lambda t: analizar_clausula.invoke(
            {"texto_clausula": t}
        ),
        "Alerta de Renovacion": lambda t: alerta_renovacion.invoke(
            {"metadata_contrato": t}
        ),
        "Estrategia de Negociacion": lambda t: negociar.invoke(
            {"contexto_contrato": t}
        ),
    }

    funcion = mapa_herramientas.get(herramienta)
    if funcion is None:
        return f"Herramienta no reconocida: {herramienta}"

    return funcion(texto)


def cb_comparar(texto_a: str, texto_b: str) -> str:
    """Callback para la pestaña 'Comparar Propuestas'.

    Args:
        texto_a: Texto de la propuesta A.
        texto_b: Texto de la propuesta B.

    Returns:
        Resultado de la comparación generada por el modelo.
    """
    if not texto_a.strip() or not texto_b.strip():
        return "Ingrese ambas propuestas para poder comparar."

    return comparar_propuestas.invoke({
        "propuesta_a": texto_a.strip(),
        "propuesta_b": texto_b.strip(),
    })


def cb_consulta_libre(archivo, pregunta: str) -> str:
    """Callback para la pestaña 'Consulta Libre'.

    Combina el contenido del archivo (si existe) con la pregunta del usuario
    y ejecuta el agente ReAct completo.

    Args:
        archivo: Archivo subido (.txt o .pdf), puede ser None.
        pregunta: Pregunta o instrucción del usuario.

    Returns:
        Respuesta final del agente ReAct.
    """
    texto_archivo: str = extraer_texto_archivo(archivo)
    consulta: str = pregunta.strip()

    if texto_archivo and consulta:
        consulta = (
            f"{consulta}\n\n"
            f"--- Contenido del contrato ---\n{texto_archivo}"
        )
    elif texto_archivo:
        consulta = (
            f"Analiza el siguiente contrato e identifica los puntos "
            f"mas relevantes para un CTO:\n\n{texto_archivo}"
        )

    if not consulta:
        return "Ingrese una pregunta o suba un archivo para analizar."

    resultado = ejecutor_agente.invoke({"input": consulta})
    return resultado["output"]


# ---------------------------------------------------------------------------
# Construcción de la interfaz Gradio con pestañas
# ---------------------------------------------------------------------------

with gr.Blocks(
    title="Agente de Contratos para CTOs",
    theme=gr.themes.Soft(),
) as demo:

    gr.Markdown(
        "# Agente de Inteligencia Contractual para CTOs\n"
        "Suba un contrato (.txt o .pdf) o pegue el texto para analizarlo "
        "con el modelo fine-tuneado."
    )

    # --- Pestaña 1: Analizar Contrato ---
    with gr.Tab("Analizar Contrato"):
        gr.Markdown(
            "Suba un archivo o pegue el texto de una clausula contractual. "
            "Seleccione la herramienta de analisis."
        )
        with gr.Row():
            with gr.Column():
                entrada_archivo = gr.File(
                    label="Subir contrato (.txt o .pdf)",
                    file_types=[".txt", ".pdf"],
                )
                entrada_texto = gr.Textbox(
                    label="O pegue el texto del contrato aqui",
                    lines=10,
                    placeholder="Clausula 8.1 - Renovacion Automatica: ...",
                )
                selector_herramienta = gr.Dropdown(
                    label="Herramienta de analisis",
                    choices=[
                        "Analizar Clausula",
                        "Alerta de Renovacion",
                        "Estrategia de Negociacion",
                    ],
                    value="Analizar Clausula",
                )
                boton_analizar = gr.Button(
                    "Analizar",
                    variant="primary",
                )
            with gr.Column():
                salida_analisis = gr.Textbox(
                    label="Resultado del analisis",
                    lines=20,
                    show_copy_button=True,
                )

        boton_analizar.click(
            fn=cb_analizar,
            inputs=[entrada_archivo, entrada_texto, selector_herramienta],
            outputs=salida_analisis,
        )

    # --- Pestaña 2: Comparar Propuestas ---
    with gr.Tab("Comparar Propuestas"):
        gr.Markdown(
            "Pegue el texto de dos propuestas de proveedores para obtener "
            "una comparacion detallada con puntajes por dimension."
        )
        with gr.Row():
            texto_propuesta_a = gr.Textbox(
                label="Propuesta A",
                lines=10,
                placeholder="Proveedor: AWS\nCosto anual: USD 480,000\n...",
            )
            texto_propuesta_b = gr.Textbox(
                label="Propuesta B",
                lines=10,
                placeholder="Proveedor: Azure\nCosto anual: USD 520,000\n...",
            )

        boton_comparar = gr.Button(
            "Comparar Propuestas",
            variant="primary",
        )
        salida_comparacion = gr.Textbox(
            label="Resultado de la comparacion",
            lines=20,
            show_copy_button=True,
        )

        boton_comparar.click(
            fn=cb_comparar,
            inputs=[texto_propuesta_a, texto_propuesta_b],
            outputs=salida_comparacion,
        )

    # --- Pestaña 3: Consulta Libre (Agente ReAct) ---
    with gr.Tab("Consulta Libre (Agente)"):
        gr.Markdown(
            "Haga cualquier pregunta sobre contratos tecnologicos. "
            "El agente ReAct decidira que herramientas usar automaticamente. "
            "Opcionalmente suba un contrato como contexto."
        )
        with gr.Row():
            with gr.Column():
                archivo_libre = gr.File(
                    label="Subir contrato como contexto (opcional)",
                    file_types=[".txt", ".pdf"],
                )
                pregunta_libre = gr.Textbox(
                    label="Su pregunta o instruccion",
                    lines=5,
                    placeholder=(
                        "Ej: Tenemos un contrato con AWS que vence en 45 dias. "
                        "Analiza los riesgos y genera una estrategia de "
                        "negociacion..."
                    ),
                )
                boton_consultar = gr.Button(
                    "Consultar al Agente",
                    variant="primary",
                )
            with gr.Column():
                salida_agente = gr.Textbox(
                    label="Respuesta del agente",
                    lines=20,
                    show_copy_button=True,
                )

        boton_consultar.click(
            fn=cb_consulta_libre,
            inputs=[archivo_libre, pregunta_libre],
            outputs=salida_agente,
        )

    gr.Markdown(
        "---\n"
        "Este analisis es asistido por IA y no constituye asesoria legal profesional."
    )


# Lanzar la interfaz con link público para compartir
demo.launch(share=True, debug=False)
print("Interfaz Gradio iniciada. Use el link publico para compartir la demo.")